# 06 — What the loss cannot see

**Series:** field-controlled generation in 2D · investigation notebook

Not part of the presentation path (00–05). This one exists to answer a question raised by
notebook 03 and to make sure the answer is *measured* rather than argued.

---

### The puzzle

Notebook 03 established two things that sit awkwardly together.

**(1) The fine-tuning is necessary.** Fed the same field with the same seed, the pretrained
scalar-conditioned model does not follow it; the fine-tuned one does. Same architecture,
same noise — only the weights differ.

**(2) The fine-tuning barely registers as learning.**

| run | val_loss first → min | drop | train_loss (smoothed) |
|---|---|---|---|
| scalar pretrain | 0.1024 → 0.0522 | 49% over 15.4k steps | 0.160 → 0.054 |
| field fine-tune | 0.0571 → 0.0513 | 10% over 7.7k steps | 0.0564 → **0.0549** (2.6%) |

The fine-tune's *first* validation already sits at the pretrained model's converged level.
The weights move by only $\lVert\Delta\theta\rVert/\lVert\theta\rVert = 0.075$ — and, against
intuition, **least of all in the conditioning embedder** (0.017), with ~63% of the drift in
the UNet bottleneck blocks.

### The hypothesis

The denoising objective is

$$\mathcal{L} = \mathbb{E}\,\lambda(\sigma)\bigl\lVert D_\theta(z_\sigma,\sigma,y) - z\bigr\rVert^2 .$$

Almost all of that error is **irreducible**: at a given $\sigma$ the noise cannot be removed.
Conditioning only helps predict the large-scale, low-frequency part of $z$ — a small share of
its variance. If that share is a couple of percent, then a loss curve is nearly blind to
conditioning adherence, and its flatness tells us nothing about *when* the behaviour was
acquired. Early stopping on such a loss would be steering by an instrument that is not
pointed at the thing we care about.

That is a testable claim, and this notebook tests it by **training with behavioural probes
running alongside the loss**, through both phases:

| section | what it does |
|---|---|
| §2 | the probe suite: four measurements the loss curve does not give you |
| §3 | a Lightning callback that runs them at log-spaced steps during training |
| §4 | **Phase A** — pretraining from scratch, tracked. When does *scalar* control appear? |
| §5 | **Phase B** — field fine-tuning, tracked densely early. When does *field* adherence appear? |
| §6 | **Phase C** — control: fine-tune on *mismatched* fields. Separates field-learning from generic drift. |
| §7 | synthesis: loss vs behaviour on shared axes, and where early stopping actually fires |
| §8 | bonus: the effective conditioning radius $\tau$, measured |

---
## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import copy
import time
import collections

import numpy as np
import scipy.stats
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import lightning
import lightning.pytorch.callbacks as pl_callbacks

import diffsci2.data
import diffsci2.models
import diffsci2.nets

import fcconfig as fc

fc.ensure_dirs()
torch.manual_seed(0)
np.random.seed(0)

DEVICE = fc.DEVICE
INVESTIGATION_DIR = os.path.join(fc.DATA_DIR, 'investigation')
LOG_DIR = os.path.join(INVESTIGATION_DIR, 'logs')
os.makedirs(LOG_DIR, exist_ok=True)
print(f"device = {DEVICE}")
print(f"logs   = {LOG_DIR}")

### Logging

Every probe point is appended to a **JSONL** file the moment it is measured, and flushed.
That means the run is readable *while it is still running* — by us, by an agent, by anything
that can read a file — and re-analysable afterwards without retraining. Alongside it:

```
fcdata/investigation/logs/
  manifest.json              every knob, resolved, plus the environment
  probes_<phase>.jsonl       one JSON object per probe point (incl. per-sigma arrays)
  progress_<phase>.log       the human-readable running commentary
  summary_<phase>.csv        flat table, scalars only
  records_all.json           everything, written at the end
```

In [ ]:
# ---------------------------------------------------------------- budgets ----
# These dominate the runtime. Defaults are set so the whole notebook is a
# ~1-2 h run; raise them for a more definitive answer.

RUN_PHASE_A = True          # pretraining from scratch, tracked
RUN_PHASE_B = True          # field fine-tuning, tracked
RUN_PHASE_C = True          # control: fine-tuning on mismatched fields

PRETRAIN_STEPS = 6000       # nb02 used 15360
FINETUNE_STEPS = 2000       # nb03 used 7680; the interesting part is the first few hundred
BATCH_SIZE = 16

# Early stopping (as a real callback; §7 also re-derives it post hoc for several patiences)
ES_PATIENCE = 6
ES_MIN_DELTA = 1e-4
VAL_EVERY = 200             # optimizer steps between validations

# Probe cadence: dense early, sparse late -- the hypothesis is that behaviour
# saturates long before the loss does.
def log_spaced_steps(total, n=26, first=10):
    s = np.unique(np.round(np.geomspace(first, total, n)).astype(int))
    return sorted(set([0] + s.tolist()))

PROBE_STEPS_A = log_spaced_steps(PRETRAIN_STEPS)
PROBE_STEPS_B = log_spaced_steps(FINETUNE_STEPS, n=30, first=5)
print(f"phase A probes at {len(PROBE_STEPS_A)} steps: {PROBE_STEPS_A[:8]} ... {PROBE_STEPS_A[-3:]}")
print(f"phase B probes at {len(PROBE_STEPS_B)} steps: {PROBE_STEPS_B[:8]} ... {PROBE_STEPS_B[-3:]}")

In [ ]:
import json
import platform
import subprocess

def write_manifest():
    '''Everything needed to reproduce or reinterpret this run, in one file.'''
    try:
        commit = subprocess.check_output(
            ['git', '-C', fc.REPO_ROOT, 'rev-parse', 'HEAD'], text=True).strip()
        dirty = bool(subprocess.check_output(
            ['git', '-C', fc.REPO_ROOT, 'status', '--porcelain'], text=True).strip())
    except Exception:
        commit, dirty = None, None

    man = {
        'notebook': '06-what-the-loss-cannot-see',
        'started': time.strftime('%Y-%m-%d %H:%M:%S'),
        'git': {'commit': commit, 'dirty': dirty},
        'env': {'python': platform.python_version(), 'torch': torch.__version__,
                'device': DEVICE,
                'gpu': torch.cuda.get_device_name(int(DEVICE.split(':')[1]))
                if torch.cuda.is_available() else None},
        'budgets': {'RUN_PHASE_A': RUN_PHASE_A, 'RUN_PHASE_B': RUN_PHASE_B,
                    'RUN_PHASE_C': RUN_PHASE_C, 'PRETRAIN_STEPS': PRETRAIN_STEPS,
                    'FINETUNE_STEPS': FINETUNE_STEPS, 'BATCH_SIZE': BATCH_SIZE},
        'early_stopping': {'patience': ES_PATIENCE, 'min_delta': ES_MIN_DELTA,
                           'val_every_steps': VAL_EVERY},
        'probe_steps': {'A': PROBE_STEPS_A, 'B': PROBE_STEPS_B},
        'fcconfig': {k: getattr(fc, k) for k in
                     ['STONE', 'WINDOW', 'RADIUS', 'F', 'Z_DIM', 'PATCH', 'LATENT_PATCH',
                      'MODEL_CHANNELS', 'NSTEPS', 'BORDER', 'COARSE_N', 'N_SLICES',
                      'TRAIN_SLICES', 'SIGMA_MIN', 'SIGMA_MAX', 'SIGMA_DATA',
                      'INITIAL_NORM', 'VOXEL_SIZE_UM']},
        'checkpoints': {'vae': os.path.join(fc.VAE_CKPT_DIR, 'last.ckpt'),
                        'scalar': os.path.join(fc.SCALAR_CKPT_DIR, 'last.ckpt'),
                        'field': os.path.join(fc.FIELD_CKPT_DIR, 'last.ckpt')},
    }
    path = os.path.join(LOG_DIR, 'manifest.json')
    with open(path, 'w') as fh:
        json.dump(man, fh, indent=2)
    print(f"manifest -> {path}")
    return man


MANIFEST = write_manifest()

In [ ]:
# ---------------------------------------------------------------- data -------
R = fc.RADIUS
slices = fc.load_slices(mmap=False)
fields = fc.load_fields(mmap=False)
bin_c = np.ascontiguousarray(slices[:, R:-R, R:-R])
phi_c = np.ascontiguousarray(fields[:, R:-R, R:-R]).astype(np.float32)
del slices, fields

train_bin, train_phi = bin_c[:fc.TRAIN_SLICES], phi_c[:fc.TRAIN_SLICES]
val_bin, val_phi = bin_c[fc.TRAIN_SLICES:], phi_c[fc.TRAIN_SLICES:]
print(f"train {train_bin.shape}   val {val_bin.shape}")

In [ ]:
class PairedDataset(torch.utils.data.Dataset):
    '''Paired (patch, field) as in notebook 03, with two extra modes.

    mode='field'      -> y = the patch's own pooled porosity field   (correct)
    mode='scalar'     -> y = the patch's scalar porosity             (pretraining)
    mode='mismatched' -> y = a DIFFERENT patch's field               (control)
    '''

    def __init__(self, binary, field, dataset_size, mode='field',
                 patch=fc.PATCH, symmetry=None, seed=None):
        self.binary, self.field = binary, field
        self.dataset_size, self.mode = dataset_size, mode
        self.patch, self.symmetry = patch, symmetry
        self.rng = np.random.default_rng(seed)

    def __len__(self):
        return self.dataset_size

    def _draw(self):
        s = self.rng.integers(0, self.binary.shape[0])
        i = self.rng.integers(0, self.binary.shape[1] - self.patch + 1)
        j = self.rng.integers(0, self.binary.shape[2] - self.patch + 1)
        sl = (slice(i, i + self.patch), slice(j, j + self.patch))
        return self.binary[s][sl].copy(), self.field[s][sl].copy()

    def __getitem__(self, idx):
        b, p = self._draw()
        x = torch.from_numpy(b).float()
        phi = torch.from_numpy(p).float()
        if self.symmetry is not None:
            k = int(self.rng.integers(0, 8))
            x, phi = self.symmetry.apply(x, k), self.symmetry.apply(phi, k)

        if self.mode == 'scalar':
            y = {'porosity': 1.0 - x.mean()}
        else:
            if self.mode == 'mismatched':
                _, phi = self._draw()               # somebody else's field
                phi = torch.from_numpy(phi).float()
            y = {'porosity': F.avg_pool2d(phi[None, None], fc.F, fc.F)[0, 0]}
        return {'x': x.unsqueeze(0), 'y': y}


symmetry = diffsci2.data.SquareSymmetry()


def make_loaders(mode, dataset_size=2 ** 12):
    tr = PairedDataset(train_bin, train_phi, dataset_size, mode=mode, symmetry=symmetry)
    va = PairedDataset(val_bin, val_phi, dataset_size // 8, mode=mode, symmetry=None, seed=1234)
    return (torch.utils.data.DataLoader(tr, batch_size=BATCH_SIZE, shuffle=True,
                                        num_workers=4, drop_last=True, persistent_workers=True),
            torch.utils.data.DataLoader(va, batch_size=BATCH_SIZE, shuffle=False,
                                        num_workers=2, persistent_workers=True))


for m in ['scalar', 'field', 'mismatched']:
    d = PairedDataset(train_bin, train_phi, 4, mode=m)
    it = d[0]
    print(f"mode={m:<12} x {tuple(it['x'].shape)}   y {tuple(np.shape(it['y']['porosity']))}")

---
## 2. The probe suite

Four measurements, none of which a training curve gives you. Everything is fixed once —
the same latents, the same noise, the same $\sigma$ grid, the same conditioning fields —
so any change over training is a change in the *model*.

**P1 · loss sensitivity.** The weighted denoising loss under four conditionings that share
their marginals but differ in spatial information: the correct field, a *mismatched* field
(another patch's), a *flat* field at the correct mean, and none. The gap between **flat**
and **correct** is exactly the share of the loss that spatial conditioning can explain. If
this is small, the loss is blind and §1's hypothesis holds.

**P2 · conditioning gain.** A finite difference: nudge the field and see how much the
denoiser output moves, separately along a *level* direction (uniform shift) and a *spatial*
direction (zero-mean pattern). This needs no sampling and measures the derivative
$\partial D/\partial\phi$ directly — a model that ignores spatial structure has a spatial
gain near zero regardless of what its loss says.

**P3 · field adherence.** Actually generate, then compute the realised field the way
notebook 03 does (window-$w$ average, then pooled) and regress it on the requested field.
Report Pearson $\rho$ and the **slope** — slope $\approx 0$ means ignoring, $\approx 1$ means
following. Done for a real measured field and for a synthetic ramp.

**P4 · weight drift.** Per-module $\lVert\Delta\theta\rVert/\lVert\theta_0\rVert$ against the
weights at the start of the phase. Free, and it says *where* in the network the change lives.

In [ ]:
class ProbeSuite:
    '''Fixed inputs + the four probes. Construct once, call on any SIModule.'''

    def __init__(self, binary, field, reference_module, device,
                 n_batches=4, bs=8, patch=fc.PATCH, gen_size=512,
                 n_sigma=7, nsteps=11, seed=0):
        self.device, self.nsteps, self.gen_size = device, nsteps, gen_size
        rng = np.random.default_rng(seed)
        gen = torch.Generator().manual_seed(seed)

        # --- fixed (latent, field, noise) batches for P1 / P2 -----------------
        xs, ps = [], []
        for _ in range(n_batches * bs):
            s = rng.integers(0, binary.shape[0])
            i = rng.integers(0, binary.shape[1] - patch + 1)
            j = rng.integers(0, binary.shape[2] - patch + 1)
            xs.append(binary[s, i:i + patch, j:j + patch])
            ps.append(field[s, i:i + patch, j:j + patch])
        x = torch.from_numpy(np.stack(xs)).float().unsqueeze(1)
        phi = F.avg_pool2d(torch.from_numpy(np.stack(ps)).float()[:, None], fc.F, fc.F)[:, 0]

        ref = reference_module.to(device).eval()
        self.batches = []
        with torch.no_grad():
            for k in range(n_batches):
                sl = slice(k * bs, (k + 1) * bs)
                z, _ = ref.encode(x[sl].to(device), None)
                z = ref.initial_norm(z)
                noise = torch.randn(z.shape, generator=gen).to(device)
                self.batches.append((z, phi[sl].to(device), noise))

        # sigma grid = the noise levels the sampler visits
        sched = ref.create_time_schedule(n_sigma + 1).cpu().numpy()[:-1]
        self.sigmas = sched

        # a fixed zero-mean unit-std spatial pattern for the gain probe
        pat = torch.randn(phi.shape[1:], generator=gen)
        pat = F.avg_pool2d(pat[None, None], 4, 1, padding=2)[0, 0][:phi.shape[1], :phi.shape[2]]
        self.pattern = ((pat - pat.mean()) / pat.std()).to(device)

        # --- conditioning fields for P3 (generation) -------------------------
        L = gen_size // fc.F
        real = field[0, :gen_size, :gen_size]
        self.cond_real = fc.to_latent(real)
        yy, xx = np.meshgrid(np.linspace(0, 1, L), np.linspace(0, 1, L), indexing='ij')
        self.cond_ramp = (0.10 + 0.18 * 0.5 * (xx + yy)).astype(np.float32)

        self.variant_names = ['field (correct)', 'field, mismatched',
                              'flat = own mean', 'none']

    # ------------------------------------------------------------------ P1 ---
    def _variants(self, phi):
        return [
            ('field (correct)',   {'porosity': phi}),
            ('field, mismatched', {'porosity': phi.roll(1, 0)}),
            ('flat = own mean',   {'porosity': phi.mean(dim=(1, 2), keepdim=True).expand_as(phi)}),
            ('none',              None),
        ]

    @torch.no_grad()
    def loss_sensitivity(self, mod):
        mod.eval()
        out = {n: np.zeros(len(self.sigmas)) for n in self.variant_names}
        for z, phi, noise in self.batches:
            for si, s in enumerate(self.sigmas):
                t = torch.full((z.shape[0],), float(s), device=self.device)
                a = mod.config.alpha_fn(t).view(-1, 1, 1, 1)
                sg = mod.config.sigma_fn(t).view(-1, 1, 1, 1)
                zs = a * z + sg * noise
                w = mod.config.loss_weighting.weighting_function(t.view(-1, 1, 1, 1))
                for name, y in self._variants(phi):
                    d = mod.get_denoiser_output(zs, t, y=y)
                    out[name][si] += float(
                        (mod.config.loss_metric_module(d, z) * w).mean()) / len(self.batches)
        return out

    # ------------------------------------------------------------------ P2 ---
    @torch.no_grad()
    def conditioning_gain(self, mod, delta=0.02):
        '''RMS change in the denoiser output per unit change in the field.'''
        mod.eval()
        lvl = np.zeros(len(self.sigmas))
        spa = np.zeros(len(self.sigmas))
        for z, phi, noise in self.batches:
            for si, s in enumerate(self.sigmas):
                t = torch.full((z.shape[0],), float(s), device=self.device)
                a = mod.config.alpha_fn(t).view(-1, 1, 1, 1)
                sg = mod.config.sigma_fn(t).view(-1, 1, 1, 1)
                zs = a * z + sg * noise
                base = mod.get_denoiser_output(zs, t, y={'porosity': phi})
                d_l = mod.get_denoiser_output(zs, t, y={'porosity': phi + delta})
                d_s = mod.get_denoiser_output(
                    zs, t, y={'porosity': phi + delta * self.pattern})
                lvl[si] += float((d_l - base).pow(2).mean().sqrt()) / delta / len(self.batches)
                spa[si] += float((d_s - base).pow(2).mean().sqrt()) / delta / len(self.batches)
        return {'level': lvl, 'spatial': spa}

    # ------------------------------------------------------------------ P3 ---
    @torch.no_grad()
    def adherence(self, mod, seed=17):
        mod.eval()
        res = {}
        for tag, cond in [('real', self.cond_real), ('ramp', self.cond_ramp)]:
            torch.manual_seed(seed)
            img = mod.sample(1, shape=[fc.Z_DIM, *cond.shape],
                             y={'porosity': torch.tensor(cond, dtype=torch.float32)},
                             nsteps=self.nsteps, is_latent_shape=True,
                             return_latents=False, guidance=1.0)
            arr = img[0, 0].cpu().numpy()
            binar = (arr > arr.mean()).astype(np.float32)
            got = fc.field_interior(fc.realised_field(binar))
            want = fc.field_interior(cond)
            lr = scipy.stats.linregress(want.ravel(), got.ravel())
            res[f'rho_{tag}'] = float(lr.rvalue)
            res[f'slope_{tag}'] = float(lr.slope)
            res[f'phi_{tag}'] = float(1 - binar.mean())
        return res


def weight_drift(model, ref_state):
    '''Per-top-level-module relative L2 change against a reference state dict.'''
    cur = {k: v.detach().cpu() for k, v in model.state_dict().items()}
    keys = [k for k in ref_state if k in cur and ref_state[k].dtype.is_floating_point]
    g = collections.defaultdict(lambda: [0.0, 0.0])
    tot = [0.0, 0.0]
    for k in keys:
        d = float((cur[k] - ref_state[k]).pow(2).sum())
        n = float(ref_state[k].pow(2).sum())
        g[k.split('.')[0]][0] += d
        g[k.split('.')[0]][1] += n
        tot[0] += d
        tot[1] += n
    out = {f'drift/{m}': float(np.sqrt(a / max(b, 1e-12))) for m, (a, b) in g.items()}
    out['drift/global'] = float(np.sqrt(tot[0] / max(tot[1], 1e-12)))
    return out

---
## 3. A tracking callback

Runs the probes at the pre-declared steps, and records the loss at the same moments so the
two can be plotted on shared axes. Log-spaced, so the early steps — where the hypothesis
says everything happens — are sampled densely.

In [ ]:
class TrackingCallback(pl_callbacks.Callback):
    '''Run the probe suite at chosen global steps and record everything.'''

    def __init__(self, suite, probe_steps, run_adherence=True, tag=''):
        super().__init__()
        self.suite = suite
        self.probe_steps = sorted(set(probe_steps))
        self.run_adherence = run_adherence
        self.tag = tag
        self.records = []
        self.ref_state = None
        self._done = set()
        # Written incrementally and flushed, so the run is readable while it runs.
        self.jsonl_path = os.path.join(LOG_DIR, f'probes_{tag}.jsonl')
        self.text_path = os.path.join(LOG_DIR, f'progress_{tag}.log')
        open(self.jsonl_path, 'w').close()
        open(self.text_path, 'w').close()

    def _emit(self, line):
        print(line, flush=True)
        with open(self.text_path, 'a') as fh:
            fh.write(line + '\n')

    def on_train_start(self, trainer, pl_module):
        self.ref_state = {k: v.detach().cpu().clone()
                          for k, v in pl_module.model.state_dict().items()}
        self._measure(trainer, pl_module, 0)

    def on_train_batch_end(self, trainer, pl_module, outputs, batch, batch_idx):
        gs = trainer.global_step
        if gs in self.probe_steps and gs not in self._done:
            self._measure(trainer, pl_module, gs)

    def _measure(self, trainer, pl_module, step):
        self._done.add(step)
        was_training = pl_module.training
        pl_module.eval()
        t0 = time.time()
        rec = {'step': step, 'tag': self.tag}

        ls = self.suite.loss_sensitivity(pl_module)
        base = ls['field (correct)'].mean()
        for n in self.suite.variant_names:
            rec[f'loss/{n}'] = float(ls[n].mean())
        rec['loss/spatial_gap_pct'] = float(100 * (ls['flat = own mean'].mean() - base) / base)
        rec['loss/mismatch_gap_pct'] = float(100 * (ls['field, mismatched'].mean() - base) / base)
        rec['loss_by_sigma'] = {n: ls[n].tolist() for n in self.suite.variant_names}

        g = self.suite.conditioning_gain(pl_module)
        rec['gain/level'] = float(g['level'].mean())
        rec['gain/spatial'] = float(g['spatial'].mean())
        rec['gain_by_sigma'] = {k: v.tolist() for k, v in g.items()}

        if self.run_adherence:
            rec.update({f'adh/{k}': v for k, v in self.suite.adherence(pl_module).items()})

        if self.ref_state is not None:
            rec.update(weight_drift(pl_module.model, self.ref_state))

        m = trainer.callback_metrics
        for k in ('train_loss', 'val_loss'):
            if k in m:
                rec[k] = float(m[k])

        rec['probe_seconds'] = time.time() - t0
        rec['wall_clock'] = time.strftime('%H:%M:%S')
        self.records.append(rec)

        # Append + flush immediately: the log is complete up to this point on disk.
        with open(self.jsonl_path, 'a') as fh:
            fh.write(json.dumps(rec) + '\n')
            fh.flush()
            os.fsync(fh.fileno())

        if was_training:
            pl_module.train()
        self._emit(
            f"  [{self.tag} step {step:>5}] "
            f"val_loss {rec.get('val_loss', float('nan')):.5f}  "
            f"spatial_gap {rec['loss/spatial_gap_pct']:+.2f}%  "
            f"gain_spatial {rec['gain/spatial']:.4f}  "
            + (f"slope_ramp {rec.get('adh/slope_ramp', float('nan')):+.3f}  "
               f"rho_ramp {rec.get('adh/rho_ramp', float('nan')):+.3f}  "
               if self.run_adherence else "")
            + f"drift {rec.get('drift/global', 0.0):.4f}  ({rec['probe_seconds']:.0f}s)")


def run_phase(tag, mode, init_weights, max_steps, probe_steps, lr,
              warmup=200, run_adherence=True):
    '''Train one phase with tracking + early stopping. Returns (records, module, es).'''
    train_loader, val_loader = make_loaders(mode)

    model = fc.make_flow_model(conditional=True)
    if init_weights is not None:
        model.load_state_dict(init_weights)
    module = fc.make_si_module(model, autoencoder=fc.load_vae())
    module.freeze_autoencoder()

    opt = torch.optim.AdamW(module.model.parameters(), lr=lr,
                            betas=(0.9, 0.999), weight_decay=0.01, eps=1e-8)
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min(1.0, (s + 1) / warmup))
    module.set_optimizer_and_scheduler(optimizer=opt, scheduler=sched,
                                       scheduler_interval='step')

    tracker = TrackingCallback(suite, probe_steps, run_adherence=run_adherence, tag=tag)
    early = pl_callbacks.EarlyStopping(monitor='val_loss', mode='min',
                                       patience=ES_PATIENCE, min_delta=ES_MIN_DELTA,
                                       verbose=True)
    trainer = lightning.Trainer(
        max_steps=max_steps,
        default_root_dir=os.path.join(INVESTIGATION_DIR, tag),
        gradient_clip_val=1.0,
        callbacks=[diffsci2.models.NanToZeroGradCallback(), tracker, early],
        devices=[int(DEVICE.split(':')[1])],
        precision='16-mixed',
        logger=False,
        enable_checkpointing=False,
        val_check_interval=VAL_EVERY,
        check_val_every_n_epoch=None,
        enable_progress_bar=True,
        log_every_n_steps=50,
    )
    t0 = time.time()
    trainer.fit(module, train_loader, val_loader)
    msg = (f"[{tag}] finished at global_step {trainer.global_step} "
           f"in {(time.time() - t0) / 60:.1f} min  "
           f"(early stop fired: {early.stopped_epoch > 0})")
    tracker._emit(msg)

    # Flat CSV alongside the JSONL, for anything that would rather read a table.
    flat = [{k: v for k, v in r.items() if not isinstance(v, dict)} for r in tracker.records]
    if flat:
        import csv
        cols = sorted({k for r in flat for k in r})
        csv_path = os.path.join(LOG_DIR, f'summary_{tag}.csv')
        with open(csv_path, 'w', newline='') as fh:
            w = csv.DictWriter(fh, fieldnames=cols)
            w.writeheader()
            w.writerows(flat)
        print(f"  -> {csv_path}")
    return tracker.records, module, early

In [ ]:
# Build the probe suite once, off a throwaway module (it is only used for encoding
# and for reading the sigma schedule, both of which are model-independent).
_ref = fc.make_si_module(fc.make_flow_model(conditional=True), autoencoder=fc.load_vae())
suite = ProbeSuite(val_bin, val_phi, _ref, DEVICE)
del _ref
print(f"probe batches : {len(suite.batches)} x {suite.batches[0][0].shape[0]}")
print(f"sigma grid    : {np.array2string(suite.sigmas, precision=3)}")
print(f"gen size      : {suite.gen_size}^2  (latent {suite.gen_size // fc.F}^2), "
      f"{suite.nsteps} ODE steps")

In [ ]:
# How long does one probe point cost? Measure before committing to a schedule.
_m = fc.make_si_module(fc.load_flow_model(os.path.join(fc.SCALAR_CKPT_DIR, 'last.ckpt')),
                       autoencoder=fc.load_vae()).to(DEVICE).eval()
for name, fn in [('loss_sensitivity', suite.loss_sensitivity),
                 ('conditioning_gain', suite.conditioning_gain),
                 ('adherence', suite.adherence)]:
    t0 = time.time(); _ = fn(_m); print(f"  {name:<20} {time.time() - t0:5.1f}s")
total_pts = len(PROBE_STEPS_A) + 2 * len(PROBE_STEPS_B)
print(f"\n{total_pts} probe points across all phases")
del _m
torch.cuda.empty_cache()

---
## 4. Phase A — pretraining, tracked

From scratch, scalar conditioning. The question: **when during pretraining does the model
acquire scalar control**, and does anything about the *field* probes move (it should not —
the model never sees a spatially varying conditioning here).

Note the field probes are still meaningful during this phase: they are exactly the case-4
question of notebook 03, measured continuously.

In [ ]:
if RUN_PHASE_A:
    rec_A, module_A, es_A = run_phase(
        'A-pretrain', mode='scalar', init_weights=None,
        max_steps=PRETRAIN_STEPS, probe_steps=PROBE_STEPS_A, lr=2e-4)
    weights_A = {k: v.detach().cpu().clone() for k, v in module_A.model.state_dict().items()}
    torch.save(weights_A, os.path.join(INVESTIGATION_DIR, 'phaseA_weights.pt'))
else:
    weights_A = fc.load_flow_weights(os.path.join(fc.SCALAR_CKPT_DIR, 'last.ckpt'))
    rec_A, es_A = [], None
    print("phase A skipped -- using notebook 02's checkpoint as the starting point")

---
## 5. Phase B — field fine-tuning, tracked

From phase A's weights, now with field conditioning. Probes are dense over the first few
hundred steps, because that is where the hypothesis says the behaviour changes.

In [ ]:
if RUN_PHASE_B:
    rec_B, module_B, es_B = run_phase(
        'B-finetune', mode='field', init_weights=weights_A,
        max_steps=FINETUNE_STEPS, probe_steps=PROBE_STEPS_B, lr=5e-5)
else:
    rec_B, es_B = [], None

---
## 6. Phase C — the control

Same starting weights, same number of steps, same learning rate — but each patch is paired
with **another patch's** field. The conditioning is statistically identical and carries no
usable information about the target.

This separates two things that phase B confounds:

- if phase B's loss drop is reproduced here, the drop was **generic continued training**,
  not field learning;
- if the weight-drift pattern is reproduced here, the drift we attributed to the
  bottleneck was also generic.

Whatever phase B does *beyond* phase C is what learning the field actually cost.

In [ ]:
if RUN_PHASE_C:
    rec_C, module_C, es_C = run_phase(
        'C-control', mode='mismatched', init_weights=weights_A,
        max_steps=FINETUNE_STEPS, probe_steps=PROBE_STEPS_B, lr=5e-5)
else:
    rec_C, es_C = [], None

---
## 7. Synthesis — loss versus behaviour

The plot the whole notebook exists for: the training loss and the behavioural probes on
shared step axes, with the early-stopping decision marked.

In [ ]:
import pandas as pd


def load_records(tag):
    '''Read a phase back from its JSONL log -- works in a fresh kernel, mid-run.'''
    path = os.path.join(LOG_DIR, f'probes_{tag}.jsonl')
    if not os.path.exists(path):
        return []
    out = []
    with open(path) as fh:
        for line in fh:
            line = line.strip()
            if line:
                try:
                    out.append(json.loads(line))
                except json.JSONDecodeError:
                    pass          # a torn final line while the run is still writing
    return out


def to_frame(records, tag=None):
    if not records and tag:
        records = load_records(tag)
    if not records:
        return pd.DataFrame()
    return pd.DataFrame([{k: v for k, v in r.items() if not isinstance(v, dict)}
                         for r in records]).sort_values('step')


dfA = to_frame(rec_A, 'A-pretrain')
dfB = to_frame(rec_B, 'B-finetune')
dfC = to_frame(rec_C, 'C-control')

# One consolidated dump, per-sigma arrays included.
with open(os.path.join(LOG_DIR, 'records_all.json'), 'w') as fh:
    json.dump({'manifest': MANIFEST,
               'A-pretrain': rec_A or load_records('A-pretrain'),
               'B-finetune': rec_B or load_records('B-finetune'),
               'C-control': rec_C or load_records('C-control')}, fh, indent=1)
print(f"wrote {os.path.join(LOG_DIR, 'records_all.json')}")
print("files in the log directory:")
for f in sorted(os.listdir(LOG_DIR)):
    print(f"  {f:<28} {os.path.getsize(os.path.join(LOG_DIR, f)) / 1024:8.1f} KB")
for name, d in [('A pretrain', dfA), ('B finetune', dfB), ('C control', dfC)]:
    if len(d):
        cols = [c for c in ['step', 'val_loss', 'loss/spatial_gap_pct',
                            'gain/spatial', 'adh/slope_ramp', 'adh/rho_ramp',
                            'drift/global'] if c in d]
        print(f"\n--- {name} ---")
        print(d[cols].to_string(index=False, float_format=lambda v: f"{v:9.4f}"))

In [ ]:
def early_stop_step(df, patience, min_delta=ES_MIN_DELTA, col='val_loss'):
    '''Where EarlyStopping(patience) would have fired, from the recorded trace.'''
    if col not in df or df[col].isna().all():
        return None
    v = df[[('step'), col]].dropna().values
    best, bad = np.inf, 0
    for step, x in v:
        if x < best - min_delta:
            best, bad = x, 0
        else:
            bad += 1
            if bad >= patience:
                return int(step)
    return None


panels = [('B-finetune (field)', dfB), ('C-control (mismatched)', dfC)]
panels = [(n, d) for n, d in panels if len(d)]

fig, axes = plt.subplots(len(panels), 3, figsize=(17, 4.6 * len(panels)), squeeze=False)
for r, (name, d) in enumerate(panels):
    s = d['step'].values

    ax = axes[r, 0]
    if 'train_loss' in d:
        ax.plot(s, d['train_loss'], 'o-', color='C7', ms=3, lw=1.4, label='train loss')
    if 'val_loss' in d:
        ax.plot(s, d['val_loss'], 's-', color='k', ms=3, lw=1.6, label='val loss')
    ax.set_xscale('symlog'); ax.set_xlabel('step'); ax.set_ylabel('denoising loss')
    ax.set_title(f"{name}\nwhat you normally watch"); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[r, 1]
    if 'adh/slope_ramp' in d:
        ax.plot(s, d['adh/slope_ramp'], 'o-', color='C0', lw=1.8, ms=4, label='slope (ramp)')
    if 'adh/rho_ramp' in d:
        ax.plot(s, d['adh/rho_ramp'], 's-', color='C2', lw=1.8, ms=4, label=r'$\rho$ (ramp)')
    if 'adh/rho_real' in d:
        ax.plot(s, d['adh/rho_real'], '^--', color='C1', lw=1.4, ms=4, label=r'$\rho$ (real field)')
    ax.axhline(1.0, color='gray', lw=0.8, ls=':')
    ax.axhline(0.0, color='gray', lw=0.8)
    ax.set_xscale('symlog'); ax.set_xlabel('step'); ax.set_ylabel('field adherence')
    ax.set_title('what you actually care about'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

    ax = axes[r, 2]
    if 'gain/spatial' in d:
        ax.plot(s, d['gain/spatial'], 'o-', color='C0', lw=1.8, ms=4, label='spatial gain')
    if 'gain/level' in d:
        ax.plot(s, d['gain/level'], 's--', color='C7', lw=1.4, ms=4, label='level gain')
    ax2 = ax.twinx()
    if 'loss/spatial_gap_pct' in d:
        ax2.plot(s, d['loss/spatial_gap_pct'], '^-', color='C3', lw=1.8, ms=4,
                 label='loss penalty for flat field [%]')
        ax2.set_ylabel('loss penalty [%]', color='C3')
        ax2.tick_params(axis='y', labelcolor='C3')
    ax.set_xscale('symlog'); ax.set_xlabel('step'); ax.set_ylabel(r'$\|\partial D/\partial\phi\|$')
    ax.set_title('sensitivity to the field'); ax.legend(fontsize=8, loc='lower right')
    ax.grid(alpha=0.3)

    for ax_ in axes[r]:
        for pat, col in [(3, 'C3'), (ES_PATIENCE, 'C1')]:
            st = early_stop_step(d, pat)
            if st is not None:
                ax_.axvline(st, color=col, ls='--', lw=1.4, alpha=0.8)

fig.suptitle("Dashed verticals: where EarlyStopping(val_loss) fires "
             f"(patience 3 and {ES_PATIENCE})", y=1.0)
fig.tight_layout()
fc.savefig(fig, "06_loss_vs_behaviour")
plt.show()

In [ ]:
# Phase B against its control, on the quantities that matter.
if len(dfB) and len(dfC):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    for ax, col, lab in [
        (axes[0], 'val_loss', 'validation denoising loss'),
        (axes[1], 'gain/spatial', r'spatial gain $\|\partial D/\partial\phi\|$'),
        (axes[2], 'drift/global', r'$\|\Delta\theta\|/\|\theta_0\|$'),
    ]:
        if col in dfB:
            ax.plot(dfB['step'], dfB[col], 'o-', color='C0', lw=2, ms=4, label='B: correct fields')
        if col in dfC:
            ax.plot(dfC['step'], dfC[col], 's--', color='C3', lw=2, ms=4, label='C: mismatched fields')
        ax.set_xscale('symlog'); ax.set_xlabel('step'); ax.set_ylabel(lab)
        ax.legend(fontsize=9); ax.grid(alpha=0.3)
    axes[0].set_title('does the loss even distinguish them?')
    axes[1].set_title('does the model become field-sensitive?')
    axes[2].set_title('is the weight drift field-specific?')
    fig.suptitle("Phase B vs its control: everything identical but the information in $\\phi$", y=1.02)
    fig.tight_layout()
    fc.savefig(fig, "06_control_comparison")
    plt.show()

In [ ]:
# Where the drift lives, over training -- and whether the control reproduces it.
mods = [c.split('/', 1)[1] for c in dfB.columns if c.startswith('drift/') and c != 'drift/global'] \
    if len(dfB) else []
if mods and len(dfB):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4.6), sharey=True)
    for ax, (d, name) in zip(axes, [(dfB, 'B: correct fields'), (dfC, 'C: mismatched fields')]):
        if not len(d):
            continue
        order = sorted(mods, key=lambda m: -d[f'drift/{m}'].iloc[-1])
        for m in order:
            ax.plot(d['step'], d[f'drift/{m}'], lw=1.7, label=m)
        ax.plot(d['step'], d['drift/global'], 'k--', lw=2.2, label='global')
        ax.set_xscale('symlog'); ax.set_yscale('log')
        ax.set_xlabel('step'); ax.set_title(name); ax.grid(alpha=0.3, which='both')
    axes[0].set_ylabel(r'$\|\Delta\theta_m\|/\|\theta_{0,m}\|$')
    axes[1].legend(fontsize=7, ncol=2, loc='lower right')
    fig.suptitle("Where the fine-tune changes the network", y=1.02)
    fig.tight_layout()
    fc.savefig(fig, "06_weight_drift")
    plt.show()

In [ ]:
# Phase A: when is *scalar* control acquired, and do the field probes stay flat?
if len(dfA):
    fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
    axes[0].plot(dfA['step'], dfA['val_loss'], 's-', color='k', lw=1.8, ms=4, label='val loss')
    axes[0].set_ylabel('denoising loss'); axes[0].set_title('pretraining loss')
    for ax, cols, title in [
        (axes[1], ['gain/level', 'gain/spatial'], 'conditioning sensitivity'),
        (axes[2], ['adh/rho_real', 'adh/rho_ramp', 'adh/slope_ramp'], 'field adherence (case 4)'),
    ]:
        for c in cols:
            if c in dfA:
                ax.plot(dfA['step'], dfA[c], 'o-', lw=1.7, ms=4, label=c.split('/')[-1])
        ax.set_title(title); ax.legend(fontsize=8)
    for ax in axes:
        ax.set_xscale('symlog'); ax.set_xlabel('step'); ax.grid(alpha=0.3)
    axes[2].axhline(0, color='gray', lw=0.8)
    fig.suptitle("Phase A: the model never sees a varying field, so field adherence should stay flat",
                 y=1.02)
    fig.tight_layout()
    fc.savefig(fig, "06_phaseA_pretraining")
    plt.show()

In [ ]:
# The headline numbers, stated rather than eyeballed.
def saturation_step(d, col, frac=0.9):
    '''First step at which `col` reaches `frac` of its final value.'''
    if col not in d or d[col].isna().all():
        return None
    v = d[[('step'), col]].dropna().values
    target = frac * v[-1, 1]
    for step, x in v:
        if (x >= target if v[-1, 1] > 0 else x <= target):
            return int(step)
    return None


print("PHASE B (field fine-tuning)")
if len(dfB):
    for col, lab in [('adh/slope_ramp', 'ramp slope'), ('adh/rho_ramp', 'ramp rho'),
                     ('gain/spatial', 'spatial gain')]:
        st = saturation_step(dfB, col)
        if st is not None:
            print(f"  {lab:<16} reaches 90% of its final value at step {st}")
    if 'val_loss' in dfB:
        v = dfB['val_loss'].dropna()
        print(f"  val_loss  {v.iloc[0]:.5f} -> {v.min():.5f} "
              f"({100 * (v.iloc[0] - v.min()) / v.iloc[0]:.1f}% drop)")
    print(f"  loss penalty for a flat field, final: "
          f"{dfB['loss/spatial_gap_pct'].iloc[-1]:+.2f}%")
    print(f"  loss penalty for a mismatched field, final: "
          f"{dfB['loss/mismatch_gap_pct'].iloc[-1]:+.2f}%")
    for pat in (3, ES_PATIENCE, 10):
        print(f"  EarlyStopping(patience={pat}) would fire at step {early_stop_step(dfB, pat)}")

---
## 8. Bonus — the effective conditioning radius

The framework's first assumption is **conditional locality**: the microstructure in a
region depends on $\phi$ only within a finite context $\tau$. That is stated in the thesis
as a modelling hypothesis and is never measured. It can be.

Perturb the conditioning field in **one latent cell**, and look at how far away the denoiser
output moves. The decay length of that response *is* $\tau$, in latent units.

In [ ]:
@torch.no_grad()
def locality_profile(mod, suite, delta=0.15, sigma_idx=2, max_r=None):
    '''RMS change in D as a function of distance from a single perturbed field cell.'''
    mod.eval()
    z, phi, noise = suite.batches[0]
    H, W = phi.shape[-2:]
    ci, cj = H // 2, W // 2
    s = float(suite.sigmas[sigma_idx])
    t = torch.full((z.shape[0],), s, device=suite.device)
    a = mod.config.alpha_fn(t).view(-1, 1, 1, 1)
    sg = mod.config.sigma_fn(t).view(-1, 1, 1, 1)
    zs = a * z + sg * noise

    base = mod.get_denoiser_output(zs, t, y={'porosity': phi})
    bumped = phi.clone()
    bumped[:, ci, cj] += delta
    pert = mod.get_denoiser_output(zs, t, y={'porosity': bumped})

    diff = (pert - base).pow(2).mean(dim=1).sqrt()          # [B, H*F, W*F] in latent px
    diff = F.adaptive_avg_pool2d(diff[:, None], (H, W))[:, 0].mean(0).cpu().numpy()

    yy, xx = np.meshgrid(np.arange(H), np.arange(W), indexing='ij')
    rr = np.sqrt((yy - ci) ** 2 + (xx - cj) ** 2)
    max_r = max_r or int(min(ci, cj))
    bins = np.arange(0, max_r + 1)
    prof = np.array([diff[(rr >= b) & (rr < b + 1)].mean() for b in bins])
    return bins, prof / max(prof[0], 1e-12), s


models_for_locality = []
if RUN_PHASE_B:
    models_for_locality.append(('post-trained (field)', module_B.to(DEVICE)))
if RUN_PHASE_C:
    models_for_locality.append(('control (mismatched)', module_C.to(DEVICE)))
models_for_locality.append(
    ('pretrained (scalar)',
     fc.make_si_module(fc.load_flow_model(os.path.join(fc.SCALAR_CKPT_DIR, 'last.ckpt')),
                       autoencoder=fc.load_vae()).to(DEVICE).eval()))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
for name, m in models_for_locality:
    bins, prof, s = locality_profile(m, suite)
    axes[0].plot(bins, prof, 'o-', lw=1.8, ms=4, label=name)
    axes[1].semilogy(bins, np.maximum(prof, 1e-6), 'o-', lw=1.8, ms=4, label=name)
for ax in axes:
    ax.set_xlabel(r'distance from the perturbed cell [latent units]')
    ax.set_ylabel('normalised response')
    ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_title(rf'response to a single-cell field bump ($\sigma$ = {s:.2f})')
axes[1].set_title('same, log scale — the decay length is $\\tau$')
fig.tight_layout()
fc.savefig(fig, "06_conditioning_locality")
plt.show()

print("1 latent unit = 8 pixels = "
      f"{8 * fc.VOXEL_SIZE_UM:.1f} um; the averaging window w = {fc.WINDOW} px "
      f"= {fc.WINDOW / fc.F:.1f} latent units")

---
## 9. What to conclude

Fill this in from the numbers above — the point of the notebook is that these are now
*measured* quantities rather than plausible stories. The three questions it was built to
settle:

1. **Is the denoising loss blind to conditioning?** Read `loss/spatial_gap_pct`: the
   percentage by which the loss worsens when the correct field is replaced by a flat field
   at the same mean. If it is small, a flat fine-tuning curve carries no information about
   field adherence, and early stopping on `val_loss` is steering by the wrong instrument.

2. **When is the behaviour actually acquired?** Compare the saturation step of
   `adh/slope_ramp` and `gain/spatial` against where `val_loss` stops improving, and against
   where `EarlyStopping` fires. A large separation is the whole finding.

3. **Is the fine-tune learning the field, or just training more?** Phase C is the control.
   Anything phase B does that phase C reproduces is not field learning — including, possibly,
   the bottleneck-concentrated weight drift measured in notebook 03.

### If the hypothesis holds, the practical consequences are

- **Do not early-stop, or model-select, on the denoising loss** when conditioning adherence
  is the objective. Select on a behavioural probe. `gain/spatial` is nearly free — it needs
  no sampling — and is a reasonable monitor to log during any conditioned diffusion training.
- **The fine-tune can be much shorter than it is.** If adherence saturates in a few hundred
  steps, the 20 epochs of the thesis recipe are mostly spent on a loss that has already
  converged. Worth checking at 3D scale, where those epochs are expensive.
- **A converged loss is not evidence of a converged conditional model.** This applies to the
  3D pipeline exactly as it does here.